# Homework 3

## Maxwell Bederman

In [1]:
from pathlib import Path

import numpy as np
import pandas as pd
from sklearn.utils import resample

## 1. Time Series Classification Part 1

### b)

In [2]:
def load_arems(data_root="../data/AReM"):
    columns = [
        "time",
        "avg_rss12", "var_rss12",
        "avg_rss13", "var_rss13",
        "avg_rss23", "var_rss23",
    ]

    root = Path(data_root)
    train_dfs, test_dfs = [], []

    for activity_dir in sorted(path for path in root.iterdir() if path.is_dir()):
        activity = activity_dir.name
        test_ids = {1, 2} if activity in {"bending1", "bending2"} else {1, 2, 3}

        csv_files = sorted(
            activity_dir.glob("dataset*.csv"),
            key=lambda path: int(path.stem.removeprefix("dataset")),
        )

        for path in csv_files:
            dataset_id = int(path.stem.removeprefix("dataset"))

            df = pd.read_csv(
                path,
                skiprows=5,
                header=None,
                names=columns,
                sep=r"\s*,\s*|\s+",
                engine="python",
                usecols=range(7),
            )

            df["activity"] = activity
            df["dataset"] = dataset_id

            if dataset_id in test_ids:
                test_dfs.append(df)
            else:
                train_dfs.append(df)

    return train_dfs, test_dfs

In [3]:
train_dfs, test_dfs = load_arems()

### c) Feature Extraction

#### i)

Max, min, mean, median, std dev, quartiles, trend, breaking up these features over smaller timeframes within the time series, seasonality, range

https://medium.com/@adnan.mazraeh1993/time-series-features-from-basics-to-super-advanced-3589944f6a98

#### ii)



In [4]:
def extract_features(dfs):
    time_series_columns = [
        "avg_rss12", "var_rss12",
        "avg_rss13", "var_rss13",
        "avg_rss23", "var_rss23",
    ]

    rows = []

    for instance_number, df in enumerate(dfs, start=1):
        row = {"Instance": instance_number}

        for series_number, column in enumerate(time_series_columns, start=1):
            series = df[column]

            row[f"min_{series_number}"] = series.min()
            row[f"max_{series_number}"] = series.max()
            row[f"mean_{series_number}"] = series.mean()
            row[f"median_{series_number}"] = series.median()
            row[f"std_{series_number}"] = series.std()
            row[f"q1_{series_number}"] = series.quantile(0.25)
            row[f"q3_{series_number}"] = series.quantile(0.75)

        rows.append(row)

    return pd.DataFrame(rows)

In [5]:
train_df, test_df = extract_features(train_dfs), extract_features(test_dfs)
total_df = pd.concat([train_df, test_df], axis=0, ignore_index=True)
total_df

,Instance,min_1,max_1,mean_1,median_1,std_1,q1_1,q3_1,min_2,max_2,...,std_5,q1_5,q3_5,min_6,max_6,mean_6,median_6,std_6,q1_6,q3_6
0,1,35.00,47.40,43.954500,44.330,1.558835,43.00,45.0000,0.0,1.70,...,1.999604,35.3625,36.500,0.0,1.79,0.493292,0.430,0.513506,0.000,0.940
1,2,33.00,47.75,42.179812,43.500,3.670666,39.15,45.0000,0.0,3.00,...,3.849448,30.4575,36.330,0.0,2.18,0.613521,0.500,0.524317,0.000,1.000
2,3,33.00,45.75,41.678063,41.750,2.243490,41.33,42.7500,0.0,2.83,...,2.411026,28.4575,31.250,0.0,1.79,0.383292,0.430,0.389164,0.000,0.500
3,4,37.00,48.00,43.454958,43.250,1.386098,42.50,45.0000,0.0,1.58,...,2.488862,22.2500,24.000,0.0,5.26,0.679646,0.500,0.622534,0.430,0.870
4,5,36.25,48.00,43.969125,44.500,1.618364,43.31,44.6700,0.0,1.50,...,3.318301,20.5000,23.750,0.0,2.96,0.555312,0.490,0.487826,0.000,0.830
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
83,15,35.50,46.25,43.174938,43.670,1.989052,42.50,44.5000,0.0,2.12,...,2.983976,12.7500,16.500,0.0,5.72,0.911979,0.830,0.666161,0.470,1.220
84,16,32.75,47.00,42.760562,44.500,3.398919,41.33,45.3725,0.0,3.34,...,4.296574,13.0000,18.565,0.0,5.73,0.842271,0.710,0.722165,0.430,1.090
85,17,19.33,43.50,34.227771,35.500,4.889576,30.50,37.7500,0.0,14.50,...,3.092094,14.7500,18.670,0.0,9.74,3.394125,3.100,1.792090,2.105,4.425
86,18,12.50,45.00,33.509729,34.125,4.850923,30.50,36.7500,0.0,13.05,...,3.133564,14.6275,18.750,0.0,8.96,3.378479,3.085,1.787360,2.060,4.440


#### iii)

In [6]:
def feature_std_summary(df, n_bootstrap=1_000, random_state=None):
    excluded_columns = {"Instance", "activity", "dataset"}

    feature_columns = [
        column
        for column in df.columns
        if column not in excluded_columns
        and pd.api.types.is_numeric_dtype(df[column])
    ]

    feature_df = df[feature_columns]
    n_instances = len(feature_df)

    if n_instances < 2:
        raise ValueError("At least two instances are required.")

    rng = np.random.default_rng(random_state)
    bootstrap_stds = []

    for _ in range(n_bootstrap):
        bootstrap_sample = resample(
            feature_df,
            replace=True,
            n_samples=n_instances,
            random_state=int(rng.integers(0, 2**32 - 1)),
        )
        bootstrap_stds.append(bootstrap_sample.std(ddof=1))

    bootstrap_stds = pd.DataFrame(bootstrap_stds)

    return pd.DataFrame({
        "feature": feature_columns,
        "sample_std": feature_df.std(ddof=1).values,
        "ci_90_lower": bootstrap_stds.quantile(0.05).values,
        "ci_90_upper": bootstrap_stds.quantile(0.95).values,
    })

In [9]:
feature_std_summary(total_df, random_state=552)

,feature,sample_std,ci_90_lower,ci_90_upper
0,min_1,9.569975,8.286940,10.667207
1,max_1,4.394362,3.334811,5.300569
2,mean_1,5.335718,4.708396,5.894766
3,median_1,5.440054,4.786177,5.990716
4,std_1,1.772153,1.577777,1.935934
5,q1_1,6.153590,5.576027,6.645973
6,q3_1,5.138925,4.337904,5.827670
7,min_2,0.000000,0.000000,0.000000
8,max_2,5.062729,4.643719,5.410571
9,mean_2,1.574164,1.411240,1.713488


#### iv)

I am choosing min, max and std dev. I think min and max are good choices because for body movements I think that the extremes make sense as an indicator of the type of movement. Also std dev because the overall variation of the movement should also be indicative of the type of movement it is.

## 2. ISLR 3.7.4

### a) 
We would expect the training RSS for the cubic model to be lower than the linear one. This is because even though the true relationship is linear, we are increasing our model complexity by adding more parameters which would overfit to the training data.

### b)
We would expect the test RSS to be lower for the linear model because the cubic model overfit to the training data and learned patterns that aren't not general to the true relationship while the linear model would be more generalized, making it perform better with unseen data.

### c)

The training RSS for the cubic model will always be lower because having the extra features gives the training process more degrees of freedom to fit to the data.

### d)

It is unclear which one would be lower in this case for the test data, it is purely based on which model is closer to the true underlying structure of the data. If the data truly has a relationship that cubic or of even higher degree, the cubic model will probably do better but in any case between linear and cubic, it is unclear because we are underfit on the linear model and overfit on the cubic model.